# 22.exe Deobfuscation Walkthrough (SPECTRALVIPER Precursor)

This notebook now includes:

- a concrete execution workflow for this sample,
- stage/offset mapping (VA and raw file offsets),
- and a practical config-extraction playbook.

All addresses are sample-specific to `22.exe` (SHA-256 `0cb5a2e3c8aa7c80c8bbfb3a5f737c75807aa0e689dd4ad0a0466d113d8a6b9d`).


In [ ]:
from pathlib import Path
import sys
import json


def find_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        new_layout = (d / 'input' / '22.exe').exists()
        old_layout = (d / '22.exe').exists()
        if (new_layout or old_layout) and (d / 'scripts').exists():
            return d
    raise RuntimeError('Could not locate SPECTRALVIPER analysis root')


ROOT = find_root(Path.cwd())
SCRIPTS_DIR = ROOT / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

ROOT


In [ ]:
from sv_analysis_lib import (
    extract_patch_bytes,
    extract_stage2,
    import_table,
    parse_pe,
    read_u32_va,
    sha256_file,
    suspicious_strings,
    write_json,
)

SAMPLE = (ROOT / 'input' / '22.exe') if (ROOT / 'input' / '22.exe').exists() else (ROOT / '22.exe')
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print('sample:', SAMPLE)
print('sample_sha256:', sha256_file(SAMPLE))


## Execution Workflow (Stage-by-Stage)

1. **Stage0 Loader (`22.exe`)** initializes anti-analysis scoring and patch primitives.
2. **Defense evasion** patches AMSI and ETW exports in-memory using `VirtualProtect` + `memcpy` + cache flush.
3. **Environment checks** score sandbox/VM/analyst indicators (Cuckoo/Sandboxie/Wine, user/host/process patterns).
4. **Embedded blob extraction** pulls encrypted payload bytes from `.data`.
5. **AES-CBC decrypt** uses in-binary key/IV to recover Stage2 PE.
6. **Stage2 triage** parses imports/strings to locate next-layer config/C2 decode routines.
7. **Config extraction (next step)** traces stage2 resolver/decrypt path, then dumps decoded config.


In [ ]:
workflow = {
    "stage0": {
        "name": "loader",
        "sample": str(SAMPLE),
        "sha256": sha256_file(SAMPLE),
    },
    "stage1_evasion": {
        "amsi_patch_fn": "sub_140002EA0 -> sub_1400041A0",
        "etw_patch_fn": "sub_140002F00 -> sub_140004270",
        "anti_sandbox_hub": "sub_140003AA0",
    },
    "stage1_to_stage2": {
        "encrypted_blob_va": hex(0x140005140),
        "encrypted_size_va": hex(0x1400A3560),
        "aes_key_va": hex(0x1400A35A0),
        "aes_iv_va": hex(0x1400A3590),
    },
    "stage2": {
        "output": str(ARTIFACTS / 'stage2_dec_unpadded.bin'),
        "purpose": "secondary PE for command/config logic",
    },
}

write_json(ARTIFACTS / 'execution_workflow.json', workflow)
workflow


## Offsets And Constants (Sample-Specific)

The table below maps important VAs to raw file offsets for this `22.exe` build.


In [ ]:
pe, data = parse_pe(SAMPLE)

constants = [
    ("AMSI patch bytes", 0x140005120, 6),
    ("Encrypted stage2 blob", 0x140005140, 0x40),
    ("Encrypted blob size dword", 0x1400A3560, 4),
    ("ETW patch bytes (primary)", 0x1400A3570, 3),
    ("ETW patch bytes (fallback)", 0x1400A3580, 3),
    ("AES IV", 0x1400A3590, 16),
    ("AES-256 key", 0x1400A35A0, 32),
]

rows = []
for name, va, size in constants:
    off = pe.va_to_off(va)
    blob = data[off:off+size] if off is not None else b''
    rows.append({
        "name": name,
        "va": hex(va),
        "raw_offset": hex(off) if off is not None else None,
        "size": size,
        "preview_hex": blob.hex(),
    })

enc_size = read_u32_va(data, pe, 0x1400A3560)
print(f"ImageBase: {hex(pe.image_base)}")
print(f"Encrypted blob size value at 0x1400A3560: 0x{enc_size:X} ({enc_size} bytes)")
rows


## Stage Extraction (Stage1 -> Stage2)

This recovers the encrypted blob, key, and IV from stage1 constants and decrypts stage2.


In [ ]:
# Sample-specific VAs for this 22.exe build
ENC_BLOB_VA = 0x140005140
ENC_SIZE_VA = 0x1400A3560
KEY_VA = 0x1400A35A0
IV_VA = 0x1400A3590

extract_report = extract_stage2(
    sample=SAMPLE,
    enc_blob_va=ENC_BLOB_VA,
    enc_size_va=ENC_SIZE_VA,
    key_va=KEY_VA,
    iv_va=IV_VA,
    out_dir=ARTIFACTS,
)
write_json(ARTIFACTS / 'stage2_extract_report.json', extract_report)
extract_report


## Stage2 Decrypt From Hex Inputs

This section lets you paste the encrypted Stage2 bytes and AES material as hex strings, decrypt them, and save Stage2.

- `ENC_BLOB_HEX`: encrypted blob bytes in hex
- `AES_KEY_HEX`: 32-byte AES key in hex
- `AES_IV_HEX`: 16-byte AES IV in hex

If left as `None`, values are auto-loaded from files generated earlier in this notebook.


In [ ]:
from sv_analysis_lib import aes256_cbc_decrypt, pkcs7_unpad
import hashlib

# Paste hex values here if you want manual input (no 0x prefix).
ENC_BLOB_HEX = None
AES_KEY_HEX = None
AES_IV_HEX = None

# Auto-fill from artifacts if manual values are not provided.
if ENC_BLOB_HEX is None:
    ENC_BLOB_HEX = (ARTIFACTS / 'enc_blob.bin').read_bytes().hex()
if AES_KEY_HEX is None:
    AES_KEY_HEX = (ARTIFACTS / 'aes_key.hex').read_text().strip()
if AES_IV_HEX is None:
    AES_IV_HEX = (ARTIFACTS / 'aes_iv.hex').read_text().strip()

enc_blob = bytes.fromhex(''.join(ENC_BLOB_HEX.split()))
key = bytes.fromhex(''.join(AES_KEY_HEX.split()))
iv = bytes.fromhex(''.join(AES_IV_HEX.split()))

if len(key) != 32:
    raise ValueError(f'AES key must be 32 bytes, got {len(key)}')
if len(iv) != 16:
    raise ValueError(f'AES IV must be 16 bytes, got {len(iv)}')

dec_raw = aes256_cbc_decrypt(enc_blob, key, iv)
dec_unpadded = pkcs7_unpad(dec_raw)

stage2_from_hex = ARTIFACTS / 'stage2_dec_from_hex.bin'
stage2_from_hex.write_bytes(dec_unpadded)

hex_decrypt_report = {
    'encrypted_blob_len': len(enc_blob),
    'encrypted_blob_sha256': hashlib.sha256(enc_blob).hexdigest(),
    'stage2_out_path': str(stage2_from_hex),
    'stage2_out_len': len(dec_unpadded),
    'stage2_out_sha256': hashlib.sha256(dec_unpadded).hexdigest(),
    'starts_with_mz': dec_unpadded[:2] == b'MZ',
}
write_json(ARTIFACTS / 'stage2_hex_input_decrypt_report.json', hex_decrypt_report)
hex_decrypt_report


## How To Extract Config (Practical Procedure)

Use this sequence for stage2 config extraction:

1. **Locate stage2 decode hub**:
   - Start from stage2 imports and suspicious strings,
   - then trace cross-references to custom decode loops / API resolvers.
2. **Identify decode boundary**:
   - Find where high-entropy or encoded blobs are transformed into low-entropy/plaintext buffers.
3. **Capture buffer after decode**:
   - Static emulation path: port the exact decode loop to Python.
   - Dynamic path: breakpoint right after decode returns, dump destination buffer.
4. **Validate config candidate**:
   - Look for URL patterns, URI paths, command tags, campaign IDs, UA/cookie tokens.
5. **Record extraction metadata**:
   - function address, input blob offset, output length, decoding key/seed/state.

For this sample, stage1 decryption offsets are fully known; stage2 config offsets are still to be derived from deeper stage2 reversing.


In [ ]:
stage2 = ARTIFACTS / 'stage2_dec_unpadded.bin'
imports = import_table(stage2)
str_hits = suspicious_strings(stage2)

config_pivot = {
    "stage2_sha256": sha256_file(stage2),
    "import_dlls": list(imports.keys()),
    "interesting_import_examples": {
        k: v[:10] for k, v in imports.items()
    },
    "suspicious_strings": str_hits,
    "next_steps": [
        "Find string/API resolver function in stage2 and map caller tree.",
        "Locate encoded blob reads in .data/.rdata and match with decode loops.",
        "Dump decoded buffer at function return for candidate config structures.",
    ],
}

write_json(ARTIFACTS / 'stage2_config_extraction_plan.json', config_pivot)
config_pivot


## AMSI / ETW Patch Extraction

These bytes are copied into target APIs after memory protection changes.


In [ ]:
patches = extract_patch_bytes(SAMPLE)
patches


## Stage2 IOC / Config Triage Snapshot


In [ ]:
stage2 = ARTIFACTS / 'stage2_dec_unpadded.bin'
imports = import_table(stage2)
hits = suspicious_strings(stage2)

ioc_report = {
    'stage2_path': str(stage2),
    'stage2_sha256': sha256_file(stage2),
    'imports': imports,
    'suspicious_strings': hits,
}
write_json(ARTIFACTS / 'stage2_ioc_report.json', ioc_report)

print('dll import count:', len(imports))
print('suspicious string count:', len(hits))
hits[:30]


## Interpretation

- Stage extraction/decryption is confirmed and reproducible with fixed offsets for this sample.
- AMSI + ETW patching and anti-sandbox scoring are strongly evidenced from code and constants.
- Final C2/config lineage remains provisional until stage2 decode routine + decoded blob are concretely recovered.
